# Data Viualisation - Analysis Report

In this phase we will analyse in details the singular features of the datasets and their correlations, and we will try to answer the questions we formulated through the use of graphic representations. Looking at graphics provides prospective on our hypothesis, allowing us to confirm or discard them.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import functions as fun

In [ ]:
countries_df = pd.read_csv('../../Data/DataFinal/countries_v2.csv')
genres_df = pd.read_csv('../../Data/DataFinal/genres_v2.csv')
languages_df = pd.read_csv('../../Data/DataFinal/languages_v2.csv')
movies_df = pd.read_csv('../../Data/DataFinal/movies_v2.csv', parse_dates=['date'], low_memory=False)
oscar_df =  pd.read_csv('../../Data/DataFinal/the_oscar_awards_v2.csv', parse_dates=['year_film','year_ceremony'],)
reviews_df = pd.read_csv('../../Data/DataFinal/rotten_tomatoes_reviews_v2.csv', parse_dates=['review_date'])

In [ ]:
plt.style.use('ggplot')

## Which Countries Made the Most Acclaimed Films?

In this first section, we would like to conduct a deeper analysis on the Countries that made the films we're studying. In particular, we will analyse the number of Oscar nominations and wins per Country, keeping in mind the chronological aspect, as well as their films' review scores, to see, not only which Nations were able to stand out the most and produce the most loved films, but also to examine any possible discrepancies between jury's and critics' opinions.

In [ ]:
countries_movies_df = fun.countries_movies_merge(countries_df, movies_df)
countries_oscar_df = fun.countries_oscar_merge(countries_movies_df, oscar_df)
countries_oscar_year_df = fun.countries_oscar_year_merge(countries_movies_df, oscar_df)
countries_reviews_df = fun.countries_reviews_merge(countries_movies_df, reviews_df)

### Which Countries have the most Oscar nominations?

In [ ]:
# Count how many times each country appears in the merged DataFrame
country_counts = countries_oscar_df['country'].value_counts()

temp_df = country_counts.reset_index()
temp_df.columns = ['country', 'count']

fig = px.choropleth(
    temp_df,
    locations="country",
    locationmode="country names",
    color="count",
    color_continuous_scale="Viridis",
    range_color=[0, 5000],
    labels={'count': 'Oscar Nominations'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

Before commenting this map, let's create one showcasing the Countries with the most Oscar Wins and then let's compare them.

### Which Countries have the most Oscar wins?

In [ ]:
# Filter only rows where 'winner' is True
winners_only_df = countries_oscar_df[countries_oscar_df['winner'] == True]
country_counts = winners_only_df['country'].value_counts()

temp_df = country_counts.reset_index()
temp_df.columns = ['country', 'count']

fig = px.choropleth(
    temp_df,
    locations="country",
    locationmode="country names",
    color="count",
    color_continuous_scale="Viridis",
    labels={'count': 'Oscar Wins'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

Now let's compare the two:

* In the first chart, we have a world map where, following the referenced scale to the right, the colour of each Country represents the number of films produced there that got nominated to the Oscars Awards.
The **USA**, in yellow, is the Country with the most nominated films, over 14 thousand, while the **United Kingdom** and **France** fall in the second and third positions respectively. While we lack data for some of them, most of the Countries have at least some nominations, even if their numbers are low, as showed by the hegemony of purple on the map.

* In the second chart, the map underlines how there is a significant gap between the first and the rest of the Countries. We can notice immediately that the **USA**, in yellow, is once again in top position, being the Country with the most Oscar-winning films. The only other Nation not coloured in purple is the **UK**, with about 900 wins, while the vast majority of the others have less than 300. Moreover, looking at the African continent, we notice that there are some Countries that, despite having some nominations, don't have any winnings.

The strong lead had by the USA shouldn't surprise us, as it is the Country that founded and hosts the Ceremony each year.

### Correlation between Oscar nominations and wins by Country:

In [ ]:
country_stats_df = fun.nom_win_number_c(countries_oscar_year_df)
country_stats_df = country_stats_df[country_stats_df['Wins'] > 0] # Include only countries with at least one win

# Interactive Bubble Plot
fig = px.scatter(country_stats_df,
                 x='Nominations',
                 y='Wins',
                 size='Nominations',
                 color='country',
                 hover_data={'Nominations': True, 'Wins': True},
                 title='Nominations vs Wins by Country')

fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title="Number of Nominations",
    yaxis_title="Number of Wins",
)
fig.show()

As we could imagine, there appears to be a clear correlation between the number of Oscar nominations and wins, with the **USA** easily being first in both categories, and the **UK** following. The relationship between nominations and wins appears to be roughly proportional for most countries, suggesting a consistent win rate.

The legend shows many Countries (Afghanistan, Albania, Algeria, etc.), though many of these smaller Countries' data points are difficult to distinguish in the crowded bottom-left section: this is why we made a dynamic chart - it's possible to uncheck, for example, the USA, to be able to zoom the view towards the other Countries.

### Which Country won the most Oscar Awards each year?

In [ ]:
countries_oscar_year_df['year_ceremony'] = pd.to_datetime(countries_oscar_year_df['year_ceremony'], errors='coerce').dt.year
winners_df = countries_oscar_year_df[countries_oscar_year_df['winner'] == True].copy()
# Get top 20 countries by total wins
top_countries = winners_df['country'].value_counts().nlargest(20).index
sorted_countries = sorted(top_countries)
filtered_df = winners_df[winners_df['country'].isin(top_countries)]
# Count wins per year per country
wins_per_year = filtered_df.groupby(['year_ceremony', 'country']).size().reset_index(name='wins')

# Plot Circle timeline
fig = px.scatter(
    wins_per_year,
    x='year_ceremony', y='country',
    size='wins', color='country',
    category_orders={'country': sorted_countries},
    title='Oscar Wins Per Year by Top 20 Countries', labels={'year_ceremony': 'Year', 'wins': 'Number of Wins'},
    width=1200, height=700
)
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(xaxis=dict(dtick=1,tickfont=dict(size=8),range=[1927, 2025]))
fig.show()

 This chart provides a visualization of the distribution of Oscar wins across multiple Countries over time. The x-axis represents the years, while the y-axis lists the top 20 Countries by total wins, each distinguished by a unique color. The size of the bubbles indicates the number of wins, with larger bubbles representing more wins in that year.

 * The **United States** consistently show a high volume of wins, particularly in recent decades, as evidenced by the dense concentration of large bubbles. This doesn't surprise us, as this Country is the one with the most Wins overall, and the one hosting the Ceremony each year.
 * Countries like **France**, the **UK** and **Germany** also exhibit notable activity, but with less consistency and fewer overall wins compared to the USA.
 * There are some Countries that show interesting behaviours: **India**, **Canada** and **New Zealand**, for example, have some periods with a lot of wins and others completely dark, with little to no activity.

### Which Countries have the films with the highest review score?

In [ ]:
country_scores = countries_reviews_df.groupby('country', as_index=False)['normalized_score'].mean()

fig = px.choropleth(
    country_scores,
    locations="country",
    locationmode="country names",
    color="normalized_score",
    color_continuous_scale="Viridis",
    labels={'normalized_score': 'Avg Score (0-10)'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

At a first glance, this map could surprise us, as it shows that there is no direct correlation between the Oscar Awards selectio and Rotten Tomatoes ratings. The above chart is, indeed, very different from the previous two: it shows some African Countries, such as **Libya**, **Somalia** and **South Sudan**, as the ones with the most appreciated movies. This could be influenced by a lower number of films produced, but we can't ignore how, overall, they are more appreciated by critics globally.

This data is interesting, as it clearly shows how the Countries with the largest film industries (such as the USA, India and China) do not necessarily have the highest average ratings. Smaller film industries in North Africa and the Middle East seem to get better reviews or audience response on average.

The average score for the **USA** is 6.23, which doesn't put it at the bottom of the scale, but is certainly far from the top. We can imagine that the USA can produce thousands of films each year, some of which deserve awards, but many of which are rather average.

**_FINAL THOUGHTS_**

Examining these charts revealed an interesting perspective on the global film industry:

The **United States** lead in Oscar nominations and wins, yet this does not necessarily reflect the quality of films produced. We saw how smaller film industries in **African** Countries, like **Libya** and **Somalia**, achieve higher average review ratings than many Hollywood films. Countries such as **Japan** demonstrate efficiency by winning more Oscars from fewer nominations, especially if we compare them to other Nations, like **Italy** for example. Moreover, Countries like **Australia** and **Sweden** have seen periods of Oscar success followed by quieter times.

Overall, this analysis helped us highlight how Oscar wins do not always align with Rotten Tomatoes critics' opinion when it comes to the ability of different Countries to produce films: the best movies, or at least the ones preferred by critics on Rotten Tomatoes, are not always the ones winning the most awards.

## Which Genres are Most Popular and Acclaimed?

In this section we would like to take a deeper look into the different genres these dataframes contain. In detail, with the chronological aspect in mind, we will once again analyse the number of Oscar nominations and wins per genre, as well as their review scores, to see which genres are usually the most appreciated and the ones able to stand out the most to the viewers, and if jury's and critics' opinions coincide this time around. Moreover, we would like to study whether the popularity of the different genres changed or stayed unvaried throughout the years.

In [ ]:
genres_movies_df = fun.genres_movies_merge(genres_df, movies_df)
genres_oscar_df = fun.genres_oscar_merge(genres_movies_df, oscar_df)
genres_countries_df = fun.genres_countries_merge(genres_movies_df, countries_df)
genres_reviews_df = fun.genres_reviews_merge(genres_movies_df, reviews_df)

### Which genres have the most Oscar nominations? And which have the most wins?

In [ ]:
top_genres = genres_oscar_df['genre'].value_counts()
#winners_only_df = genres_oscar_df[genres_oscar_df['winner'] == True]
top_genres_win = genres_oscar_df[genres_oscar_df['winner'] == True]['genre'].value_counts()

# Sort the values in descending order (largest to smallest)
top_genres = top_genres.sort_values(ascending=True)
top_genres_win = top_genres_win.sort_values(ascending=True)
viridis_colours = plt.cm.viridis(np.linspace(0, 1, 19))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
#Donut Chart 1
axes[0].pie(top_genres, labels=top_genres.index, colors=viridis_colours, startangle=-270, wedgeprops={'width': 0.5}, pctdistance=0.85,
        textprops={'fontsize': 7, 'fontweight': 'bold'})
axes[0].set_title("Top Genres by Number of Oscar Nominations", fontsize=12, fontweight='bold')
#Donut Chart2
axes[1].pie(top_genres_win, labels=top_genres_win.index, colors=viridis_colours, startangle=-270, wedgeprops={'width': 0.5}, pctdistance=0.85,
        textprops={'fontsize': 7, 'fontweight': 'bold'})
axes[1].set_title("Top Genres by Number of Oscar Wins", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

While these two plots look very similar at first glance, they actually show some meaningful differences.

**Drama** dominates both nominations and wins charts by a significant margin, represented by the large purple section in both charts, which doesn't surprise us as it also was the genre which appeared the most in out dataframes (as seen in _visualisation_). It is also followed by the **Romance**, **Comedy**, **History** and **Adventure** genres in both plots.

After that though, if we take a closer look we can notice the first differences: **War** films seem to be winning a lot more than **Thrillers**, despite the former having less nominations than the former.

If we think about how the **Romance** and **Documentary** genres closely follow **Drama** as the ones that had more entries in of datasets, we can observe an interesting fact. While we're not surprised to see **Romance** at the top of both charts, as this only highlights its great appreciation by the Oscar's jury, **Documentaries** are towards the bottom of the list, demonstrating how they're not the jury's favourites, neither when it comes to Nominations nor Wins.

Overall, for most genres the proportions appear very similar between the number of nominations and wins, suggesting that most of them convert nominations to wins at roughly similar rates. This consistency between the two charts suggests, as it could be expected, that the Academy's genre preferences remain _fairly consistent from nomination to final selection of winners_.

### Correlation between Oscar nominations and wins by genre:

In [ ]:
genre_stats_df = fun.nom_win_number_g(genres_oscar_df)

# Interactive Bubble Plot
fig = px.scatter(genre_stats_df,
                 x='Nominations',
                 y='Wins',
                 size='Nominations',
                 color='genre',
                 hover_data={'Nominations': True, 'Wins': True},
                 title='Correlation between Nominations and Wins by Genre')

fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title='Number of Nominations',
    yaxis_title='Number of Wins'
)

As we saw in the previous chart, the **Drama** genre has by far the most nominations and wins. It has around 6,000 nominations and 1,200 wins, which significantly outpaces all other genres. This reflects how critically favoured Drama is by the Oscar Awards' jury.

Genres like **Comedy**, **Romance**, **History** and **Adventure** are in the middle of the cluster, still having received a significant amount of both.

We can also notice how some genres were nominated about the same numbers of times as others, but aren't able to convert those nominations into wins as often. This is the case of **Mistery** and **Science Fiction**, and of **Crime** and **War**, outlining the under-recognition of some genres, despite commercial popularity.

### Which genres have the films with the highest review score?

In [ ]:
# We'll make 2 Violin Plots since there are so many genres
fun.genres_reviews_violin(genres_reviews_df)

These violin plots give us a general idea, but let's create one more plot to see the difference in scores more in detail.

In [ ]:
fun.genres_reviews_bar(genres_reviews_df)

As this barplot highlights, there isn't a prominent difference between the average review scores of the different genres. We can see how **Historical** movies are the genre with the highest score, closely followed by **War** and **Western**. On the contrary, critics appreciate Action films the least, but overall we can say that all genres scored similarly on the Rotten Tomatoes website, in fact the lowest-rated genres still maintain average scores of around 6 out of 10, indicating that most films across all genres tend to receive moderately positive reviews.

What surprises us is, once again, seeing that _the critics on Rotten Tomatoes don't have similar opinions to the Oscar Award's jury whatsoever_. If **Drama**, **Romance**, **Comedy** and **Documentary** were the most acclaimed genres in earlier charts, here they seem to be scattered in the middle of the plot. **Westerns** even were at the bottom of both Oscar Awards' donut charts earlier, while here this genre makes top 3 as most appreciated genre.

### How has the genres popularity changed over the years?

In [ ]:
fun.avg_score_genres(genres_reviews_df)

In this chart, we are calculating the average scores of each genre per year and looking at their trends over time.

Before taking a closer look, we can notice pretty interesting observations looking at the complex of the plot:
* There is a significant volatility in ratings across all genres throughout the entire period;
* Scores generally range between 4 and 8.5 out of 10, with some noticeable outliers;
* There appears to be a period of higher scores and greater fluctuation between roughly 1920-1960;
* In most recent years the trends tend to unify and settle around the 5.5-7 range;
* Extreme outliers (mostly very low scores) tend to appear more in earlier decades than recent years.

Concentrating on **Drama** and **History** genres, the two favoured by the Oscar Awards' jury and Rotten Tomatoes' critics respectively, we can notice that:
* Drama had a pretty linear trend over the years;
* History had some significant fluctuations in the early decades of 1900, before settling in recent years.

### Which genres are most popular in each Country?

Since the number of Countries is quite large, in the following chart we will be analyzing only the top 20 by number of films.

In [ ]:
pivot_df = fun.pivot_count_gen(genres_countries_df, reviews_df)
# Heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(pivot_df,
            cmap='viridis_r',
            annot=True,
            fmt=".2f",
            linewidths=0.5)
plt.title('Average Rotten Tomatoes Score by Country and Genre')
plt.xlabel('Genre')
plt.ylabel('Country')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In this chart, each cell represents the average Rotten Tomatoes score for films from a specific Country (rows) in a specific genre (columns).
Thanks to the colour scale, some information we can immediately gather is:
* The scores vary from a minimum of 4.50 to a maximum of 8.90, showing a great difference in opinions across genres and Countries;
* The **Philippines** has two of the highest scores, in **History** and **Fantasy** films;
* **Brazil** is the other Country that stands out the most, thanks to its high score in **Western** movies;
* The genre **Western** has a really low score in **South Korea** and **Russia**;
* **Chinese** and **Argentinian** **Musicals** aren't reviews positively often;
* Some cells appear blank (dark grey), indicating insufficient data for those Country-genre combinations.

Some of these scores can seem a bit strange, but we have to keep in mind that our dataframes only analyse a portion of all films ever made, and some interesting data can be the result of smaller productions. Therefor, we shouldn't be surprised to see the chart showing, for example, that Brazil is making better Westerns than the USA, even if in some cases this could testify that some industries produce films with a better quality than bigger ones.

Finally, this visualization demonstrates that film quality varies significantly not just by genre, but by the intersection of Country and genre, with each Nation having distinct cinematic strengths and weaknesses.

_**FINAL THOUGHTS**_

Our analysis on genres gave us some interesting prospective, in particular:

While **Drama** and **Romance** dominate the Oscar Awards, most genres receive similar ratings on the Rotten Tomatoes website, with **Historical**, **War** and **Western** as favourites. Adding to this, despite their commercial popularity, genres such as **Horror** and **Fantasy** have struggled to convert nominations into wins.

Country-genre combinations lead to unique patterns of excellence or of poor reception, partly due to the different sized industries across the different Nations and the strengths of regional filmmakers in global cinema.

Overall, this analysis helped us highlight how, just as it happens studying film producing Countries, Oscar wins do not always align with Rotten Tomatoes critics' opinion on genres: the best movies, or at least the ones preferred by critics on this website, are not always the ones winning the most awards. The same pattern can be noticed between critics and the public, where we found a persistent discrepancy between critical recognition and audience preferences.

## What is the Recipe to Make a Great Film?

After taking a deeper look into which genres and which Countries have been most successful over time, we'd like to analyse what the most acclaimed films have in common and what the key ingredients are for a film to be remembered as great.

We will now concentrate on the study of movie lengths and languages, as well as the different perceptions Rotten Tomatoes' critics and the Oscar Awards' jury have of the same films, to discover which ones are more interesting to the public and, by the end of our analysis, to get a clear summary of all the best qualities to have in order to obtain the recipie for a, in theory, successful film. We will also make sure to see if, at least on some of the best movies, the Oscar Awards' jury and Rotten Tomatoes' critics will have similar opinions or if they will dissonate once again.

### What's the correlation between Oscar Wins and reviews scores?

First of all, let's see if Rotten Tomatoes critics and the Oscar Awards jury have different perceptions on the same movies.

In [ ]:
#We are going to make 3 Heatmaps
fun.map_data_rev(oscar_df, reviews_df)

In these charts, we selected the Top 10 films that won the most Oscar Awards and the Top 10 films that have the highest average review score on the Rotten Tomatoes website, and put them side by side in 3 Heatmaps, starting from the idea that, as we saw in our earlier analysis, the two juries often have discording opinions.

We can immediately notice that, if a film has multiple Oscar nominations and wins, it usually has a great score on Rotten Tomatoes as well.

This has its sole exception in **Ben-Hur**, which, despite winning 11 Oscar prizes, scored just 5.38 on the critics' reviews. The reasons why a film like this scored so low such a low could reside in its themes and plot: in short, the film is about a Jewish prince betrayed by a Roman friend as he seeks revenge and redemption during the time of Christ. We can imagine that the Christian themes are not particularly liked by the modern global audience, and today's viewers are used to CGI-heavy, fast-paced films, while **Ben-Hur** is slow and overlong, especially compared to today’s tighter editing styles; moreover some viewers rate based on enjoyment alone, not legacy, artistry, or historical importance.

Instead, out data shows that the contrary isn't necessarily ture: it's clear that most of the Top 10 Review-Scored films don't have any Oscar Awards nominations or wins, except for just 4 of them. We also have to keep in mind that these data could not fully represent reality: for example, **Sunset Boulevard** actually was nominated 20 times at the Oscars and won many prizes, but in our dataset that dataset is missing.

Yet, the **Godfather** distinguishes itself in these charts, not only by scoring 9.71 on Rotten Tomatoes but also by being nominated in 11 categories and winning 3 of those.

Earlier in our analysis we saw how there wasn't a common opinion on which genres or which Countries made the best films, but it is further surprising to see that _the Oscar Awards' jury and Rotten Tomatoes critics don't seem to have a concordant opinion, even when it comes to the top 10 films by Oscar wins or review score_.

### Correlation between duration and public interest:

Let's check whether the films with the longest or shortest duration captured the public interest the most.

In [ ]:
duration_grouped = fun.duration_merge(movies_df, reviews_df)

# Interactive Bubble Plot
fig = px.scatter(duration_grouped,x='avg_score',y='duration',
    size='review_count',
    hover_name='original_title',
    labels={'avg_score': 'Average Score', 'duration': 'Duration (minutes)', 'review_count': 'Number of Reviews'},
    color='review_count',
    color_continuous_scale='Viridis',
    title='Correlation between Rating Score, Number of Reviews and Duration',
    width=1000,
    height=700
)
fig.update_layout(xaxis_title="Rating Score", yaxis_title="Duration")
fig.show()

The analysis of the chart provides several important insights:
* **Duration Is Mostly Between 80–130 Minutes**: most films tend to fall within the 90 to 120 minute range, indicating that this duration is generally preferred by audiences. Very few films go beyond 200 minutes.
* **Outliers in Duration**: a limited number of films surpass 500 minutes, indicating extraordinarily long films. These films don’t appear to attract many reviews or ratings, based on the size and colour of the dots.
* **Highly Reviewed Films Have Mid-to-High Ratings**: the largest and brightest bubbles are generally located around ratings of 7 to 8. This suggests that better-rated films attract more reviews.
* **Low-Rated Films (<4) Get Fewer Reviews**: there are fewer large or bright dots in the lower-left portion of the chart (low score, short/average duration), showing that poorly rated films receive less attention.

In conclusion, while _there isn’t a strong correlation between film duration and ratings_, the data suggests that moderately long films (90 to 120 minutes) paired with higher ratings (around 7 to 8) can optimize audience engagement.

### Which films aroused the most public interest?

Now we'd like to see if there are some common denominators in films that capture the public's attention the most. We can expect this to be the case with big productions and famous actors starring in these films, but let's see what our data shows us.

In [ ]:
discussed = reviews_df.groupby('movie_title').size().reset_index(name='Review Count')
# Selecting the films with the highest number of reviews
most_discussed = discussed.sort_values('Review Count', ascending=False).head(15)
most_discussed['movie_title'] = most_discussed['movie_title'].str.title()

# Interactive Bar Plot
fig_most_discussed = px.bar(most_discussed, x='Review Count', y='movie_title', orientation='h',
                            title="Top 15 Films by Number of Reviews",
                            labels={'Review Count': 'Number of Reviews', 'movie_title': 'Film Title'},
                            color='Review Count',
                            color_continuous_scale='Plasma')

fig_most_discussed.update_layout(yaxis={'categoryorder':'total ascending'},
                                 xaxis_title="Number of Reviews",
                                 yaxis_title="Film Title",
                                 margin=dict(l=150))
fig_most_discussed.show()

The chart shows the "Top 15 Films by Number of Reviews" displayed as a horizontal bar chart.
The top 3 is composed of:
* **Halloween** (approximately 405 reviews) - shown with a yellow bar. This name often times is used to refer to a trilogy of films, which could be the reason why it got so many reviews;
* **A Star Is Born** (approximately 400 reviews) - shown with a light orange bar;
* **The Lion King** (approximately 390 reviews) - shown with an orange bar.
The remaining films on the list have between approximately 320-370 reviews, with a gradient of purple colours as the number decreases.

Many popular **franchise films** appear in this list, including multiple Marvel, Star Wars, and Disney productions, suggesting these types of films tend to attract people's attention and get more reviews. This, paired with Lady Gaga starring in **A Star Is Born**, can confirm our hypothesis of the _public being more interested in productions with a big reputations_, often due to having an internationally famous celebrity on the set or being preceded by other successful films.

### What are the languages used the most in top rated films?

Finally, let's see what the Top 15 most used languages are among the top 10.000 films with the highest reviews score, so that we can get a general idea on which languages can help make a movie popular and acclaimed.

In [ ]:
fun.top_language_trend(languages_df, movies_df, reviews_df)

Earlier, during our initial data visualisation, we saw how English is the most globally used language in the films we're analysing, followed by French, Spanish, Portuguese and German.

Comparing our previous results with this chart, we can notice how **English**, **French**, **German** and **Spanish** still are at the top of the chart, as we expected, but some interesting differences can be found as well:
* **Portuguese** cannot be found among these Top 15 languages, which surprises us as it was the 4th most common earlier;
* **Xhosa**, **Somali** and **Swahili** appear in this chart but not in the previous one: these languages are primarily spoken in the African Continent, which allows us to further confirm that some African Countries have highly appreciated industries, as we saw earlier looking at the Countries with the highest Rotten Tomatoes review scores;
* Speaking of Asia, we notice **Japanese**, **Chinese** and **Korean** still being part of this chart, while **Hindu** doesn't seem to present anymore. We can imagine that, while Bollywood films are greatly appreciated in South-East Asia, these aren't as well renowned and reviewed in the rest of the world.

Finally, we can conclude that _sticking to the most used languages_ in this plot can be a good idea if we want to try to make our films known and appreciated internationally.

_**FINAL THOUGHTS**_

This final analysis we conducted on the films in these dataframes gave us some interesting prospective, in particular:

We are able to say that there isn't a sure way to tell if a film will be well received by the public and juries: too many factors concur into this, and as, we saw, the Oscar Awards' jury and Rotten Tomatoes critics never seem to have a concordant opinion, not on genres, on which Countries made the best films, nor when it comes to the top 10 films by Oscar wins or by review score.

While not being a guarantee for success, having a internationally famous actor star in a film, or said film being produced by companies with a renowned reputation, surely can help providing higher public interest and engagement.

We also confirmed our hypothesis when it comes to which language we should use: sticking to vastly known languages, such as **English**, **French**, **German** or **Spanish**, can be a good idea if we want to try to make our films known and appreciated internationally.

Finally, we also concluded that the best duration should be in the 90 to 120 minutes range, as the films around ths length are usually the ones scoring best among Rotten Tomatoes' critics.